In [1]:
import os, io, re, json, warnings
import numpy as np
import pandas as pd
import boto3

LOCAL_MODE = False

BUCKET  = "nyp-26s1-iti113"
TEAM_ID = "team14"
STUDENT_ID = "s1401"

COURSE = "ITI113"
SEMESTER = "26S1"


PROJECT_NAME01 = "airbnb-instant-booking"
PROJECT_NAME02 = "airbnb-listings"
PROJECT_NAME99 = "airbnb-recommendation"

PREFIX99  = f"iti113/{TEAM_ID}/data/{PROJECT_NAME99}"
PREFIX01  = f"iti113/{TEAM_ID}/data/{PROJECT_NAME01}"
PREFIX02  = f"iti113/{TEAM_ID}/data/{PROJECT_NAME02}"

# Initialize AWS Session & S3 Client
boto_session = boto3.Session()
region = boto_session.region_name or 'us-east-1'
s3 = boto_session.client('s3')

# Try loading SageMaker Session & Execution Role (fallback gracefully if using external IAM user/keys)
try:
    import sagemaker
    session = sagemaker.Session(boto_session=boto_session)
    role = sagemaker.get_execution_role()
except Exception:
    session = None
    role = "arn:aws:iam::default:role/LocalOrExternalRole"


# Separate output prefix for instant-booking classification
PREFIX = f"iti113/{TEAM_ID}/data/airbnb-instant-booking"
LOCAL_OUT = "processed_instant_booking"

# ---- data source ----
DATA_CANDIDATES = [
    "Listings.csv", 
    os.path.join("data", "Listings.csv"),
    "/mnt/user-data/uploads/Listings.csv"
]
DATA_PATH = next((p for p in DATA_CANDIDATES if os.path.exists(p)), DATA_CANDIDATES[0])

# ---- modelling constants (Binary Classification) ----
TARGET_COLUMN         = "instant_bookable"   # Binary target flag: 't'/'f' or 1/0
TARGET_POSITIVE_CLASS = 1                    # 1 = Instant booking enabled, 0 = Requires approval
RANDOM_STATE          = 42
TEST_SIZE             = 0.20
SNAPSHOT_DATE         = pd.Timestamp("2021-03-01")   # Data as-of date (max host_since = 2021-02-26)

# ---- classification & evaluation policy ----
DECISION_THRESHOLD    = 0.50                 # Default baseline; tune post-calibration
CALIBRATION_METHOD    = "isotonic"           # Ensures output probabilities are reliable for rankers
STRATIFY_COLUMNS      = ["city", "instant_bookable"]  # Joint stratification for regional/class balance

# ---- documented policy constants (outlier & feature engineering policies) ----
TOP_K_AMENITIES       = 30      # Amenity vocabulary size (capturing self-checkin/keypads), fit on train
COORD_DECIMALS        = 2       # Coordinate rounding: 2 dp ~ 1.1 km cells (geo-privacy)
PRICE_CAP_QUANTILE    = 0.995   # Feature-level price winsorisation per city, fit on train
COUNT_CAP_QUANTILE    = 0.99    # host_total_listings_count cap, fit on train
MIN_NIGHTS_CAP        = 365     # Domain constant: >1 year minimum stay is not nightly rental
MAX_NIGHTS_CAP        = 1125    # Platform default ceiling; larger values are sentinels
BEDROOMS_CAP          = 16      # Platform max party size; larger values are stability caps
SPARSITY_THRESHOLD    = 0.80    # Columns with >80% missing are dropped as unusable

# ---- fixed external reference points (NOT computed from data) ----
CITY_CENTERS = {                     # (lat, lon) of a canonical central landmark
    "Paris":          (48.8530,   2.3499),   # Notre-Dame
    "New York":       (40.7580, -73.9855),   # Times Square / Midtown
    "Sydney":         (-33.8568, 151.2153),  # Circular Quay
    "Rome":           (41.8986,  12.4769),   # Pantheon
    "Rio de Janeiro": (-22.9711, -43.1822),  # Copacabana
    "Istanbul":       (41.0054,  28.9768),   # Sultanahmet
    "Mexico City":    (19.4326, -99.1332),   # Zocalo
    "Bangkok":        (13.7460, 100.5340),   # Siam
    "Cape Town":      (-33.9221, 18.4231),   # City Centre
    "Hong Kong":      (22.2819, 114.1582),   # Central
}

# ---- verification output ----
print(f"Mode         : {'LOCAL (no SageMaker/S3)' if LOCAL_MODE else 'SageMaker / S3 Active'}")
print(f"Bucket       : {BUCKET}")
print(f"Prefix       : {PREFIX}")
print(f"Region       : {region}")
print(f"Role         : {role.split('/')[-1] if role else 'None'}")
print(f"Team ID      : {TEAM_ID}")
print(f"Student ID   : {STUDENT_ID}")
print(f"Task         : Binary Classification")
print(f"Target       : {TARGET_COLUMN}")
print(f"Data path    : {DATA_PATH}")
print(f"Snapshot     : {SNAPSHOT_DATE.date()}")
print(f"numpy {np.__version__} | pandas {pd.__version__}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Mode         : SageMaker / S3 Active
Bucket       : nyp-26s1-iti113
Prefix       : iti113/team14/data/airbnb-instant-booking
Region       : ap-southeast-1
Role         : SageMakerExecutionRole-ITI113-Team14
Team ID      : team14
Student ID   : s1401
Task         : Binary Classification
Target       : instant_bookable
Data path    : Listings.csv
Snapshot     : 2021-03-01
numpy 1.26.4 | pandas 2.3.3


In [2]:
# --- 1.2 Version-control the raw file in S3 (skipped gracefully in local mode)
if not LOCAL_MODE:
    RAW_S3_URI = f's3://{BUCKET}/{PREFIX99}/raw/Listings.csv'
    s3.upload_file(DATA_PATH, BUCKET, f'{PREFIX99}/raw/Listings.csv')
    print(f'Raw data uploaded to: {RAW_S3_URI}')
else:
    print('LOCAL_MODE: skipping raw S3 upload (would go to '
          f's3://{BUCKET}/{PREFIX99}/raw/Listings.csv)')

Raw data uploaded to: s3://nyp-26s1-iti113/iti113/team14/data/airbnb-recommendation/raw/Listings.csv


## Prepare the Master DataFrame

In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. Example dummy master data (replace with your merged DataFrame)
data = {
    "listing_id": [101, 102, 103, 104, 105],
    "city": ["New York", "New York", "New York", "Boston", "New York"],
    "accommodates": [2, 4, 2, 2, 2],
    "actual_price": [120, 250, 95, 110, 180],
    "pred_price": [150, 220, 100, 115, 140],  # From classmate's regression model
    "instant_book_prob": [
        0.90,
        0.40,
        0.80,
        0.95,
        0.15,
    ],  # From your classification model
}

df = pd.DataFrame(data)

# 2. Calculate Price Residual (Predicted Price - Actual Price)
# Positive residual = Deal (Underpriced relative to features)
# Negative residual = Overpriced
df["price_residual"] = df["pred_price"] - df["actual_price"]

# 3. Scale Price Residual between 0 and 1 (so it matches probability scale)
scaler = MinMaxScaler()
df["value_score"] = scaler.fit_transform(df[["price_residual"]])

## Define the Recommendation Function

In [4]:
def recommend_listings(
    df,
    city,
    min_guests=1,
    max_price=None,
    weight_value=0.5,
    weight_instant=0.5,
    top_n=5,
):
    """Filters listings based on basic criteria and re-ranks using ML outputs."""
    # Step A: Candidate Filtering (Hard Filters)
    candidates = df[
        (df["city"].str.lower() == city.lower())
        & (df["accommodates"] >= min_guests)
    ].copy()

    if max_price is not None:
        candidates = candidates[candidates["actual_price"] <= max_price]

    if candidates.empty:
        return "No listings match your search criteria."

    # Step B: Static Re-Ranking Formula
    # Final Score = (w1 * Value Score) + (w2 * Instant Book Probability)
    candidates["final_recommendation_score"] = (
        weight_value * candidates["value_score"]
    ) + (weight_instant * candidates["instant_book_prob"])

    # Step C: Sort descending and return top results
    ranked_results = candidates.sort_values(
        by="final_recommendation_score", ascending=False
    )

    columns_to_show = [
        "listing_id",
        "city",
        "accommodates",
        "actual_price",
        "value_score",
        "instant_book_prob",
        "final_recommendation_score",
    ]

    return ranked_results[columns_to_show].head(top_n)

In [5]:
# Search for: 2 guests in New York under $200
results = recommend_listings(
    df=df,
    city="New York",
    min_guests=2,
    max_price=200,
    weight_value=0.6,  # 60% emphasis on good deal/value
    weight_instant=0.4,  # 40% emphasis on instant booking convenience
    top_n=3,
)

print(results)

   listing_id      city  accommodates  actual_price  value_score  \
0         101  New York             2           120     1.000000   
2         103  New York             2            95     0.642857   
4         105  New York             2           180     0.000000   

   instant_book_prob  final_recommendation_score  
0               0.90                    0.960000  
2               0.80                    0.705714  
4               0.15                    0.060000  


## predictions on a master CSV

(Price Regression) for master CSV

In [6]:
import pandas as pd
import boto3
import json
import numpy as np

# 1. Load the raw master data (matching the regression pipeline's loading method)
# The regression pipeline uses utf-8 with encoding_errors="replace"
raw_df = pd.read_csv(
    f"s3://{BUCKET}/{PREFIX99}/raw/Listings.csv", 
    encoding="utf-8", 
    encoding_errors="replace", 
    low_memory=False
)

# 2. PREPARE THE PAYLOAD FOR THE REGRESSION ENDPOINT
# The regression pipeline's ListingFeatureEngineer handles most of the feature cleaning internally,
# but we still must replace numpy NaNs with Python None so the JSON.dumps() doesn't crash with a 500 error.
df_for_inference = raw_df.replace({np.nan: None})

# 3. Setup SageMaker runtime 
sm_runtime = boto3.client("sagemaker-runtime", region_name="ap-southeast-1")
REGRESSION_ENDPOINT = "iti113-team14-airbnb-price-sls"

def get_price_predictions_in_batches(df, batch_size=1000):
    all_probs = []
    records = df.to_dict(orient='records')
    total_records = len(records)
    
    print(f"Starting price predictions for {len(records)} records...")
    
    for i in range(0, len(records), batch_size):
        batch = records[i : i + batch_size]
        payload = json.dumps(batch)  # Note: The regression inference.py expects a list directly, not {"data": batch}
        
        response = sm_runtime.invoke_endpoint(
            EndpointName=REGRESSION_ENDPOINT,
            ContentType="application/json",
            Accept="application/json",
            Body=payload
        )
        
        # Parse the JSON response
        result = json.loads(response["Body"].read().decode("utf-8"))
        
        # Extract the 'suggested_nightly_price' from the response
        prices = [res["suggested_nightly_price"] for res in result]
        all_probs.extend(prices)
        
          # Calculate overall progress
        processed = min(i + batch_size, total_records)
        percentage = (processed / total_records) * 100
        
        print(
            f"Processed {processed:,} / {total_records:,} records "
            f"({percentage:.1f}%)"
        )
        
    return all_probs

# 4. Generate predictions and save your output
raw_df["pred_price"] = get_price_predictions_in_batches(df_for_inference)

reg_output = raw_df[['listing_id', 'pred_price']]
reg_output.to_csv('regression_predictions.csv', index=False)
print("Regression predictions saved successfully!")

# 5. UPLOAD the local file to S3 using your requested pattern

# --- 1.2 Version-control the raw file in S3 (skipped gracefully in local mode)
if not LOCAL_MODE:
    RAW_S3_URI = f's3://{BUCKET}/{PREFIX02}/raw/regression_predictions.csv'
    s3.upload_file(DATA_PATH, BUCKET, f'{PREFIX02}/raw/regression_predictions.csv')
    print(f'Raw data uploaded to: {RAW_S3_URI}')
else:
    print('LOCAL_MODE: skipping raw S3 upload (would go to '
          f's3://{BUCKET}/{PREFIX02}/raw/regression_predictions.csv)')

Starting price predictions for 279712 records...


Processed 1,000 / 279,712 records (0.4%)


Processed 2,000 / 279,712 records (0.7%)


Processed 3,000 / 279,712 records (1.1%)


Processed 4,000 / 279,712 records (1.4%)


...

Processed 278,000 / 279,712 records (99.4%)


Processed 279,000 / 279,712 records (99.7%)


Processed 279,712 / 279,712 records (100.0%)


Regression predictions saved successfully!


Raw data uploaded to: s3://nyp-26s1-iti113/iti113/team14/data/airbnb-listings/raw/regression_predictions.csv


(Binary Logistic Regression) for master CSV

In [7]:
import pandas as pd
import boto3
import json
import numpy as np

# 1. Load the raw master data locally (with the correct encoding)
raw_df = pd.read_csv(f"s3://{BUCKET}/{PREFIX99}/raw/Listings.csv", encoding="latin1", low_memory=False)

# 2. AGGRESSIVELY PRE-CLEAN BOOLEAN COLUMNS
boolean_cols = [
    'host_is_superhost', 
    'host_has_profile_pic', 
    'host_identity_verified', 
    'instant_bookable'
]

for col in boolean_cols:
    if col in raw_df.columns:
        # 1. Fill NaNs with 'f' immediately so there are no missing values
        raw_df[col] = raw_df[col].fillna('f')
        
        # 2. Map the strings to numbers
        raw_df[col] = raw_df[col].astype(str).str.lower().str.strip().map(
            {'t': 1, 'true': 1, '1': 1, 'f': 0, 'false': 0, '0': 0}
        )
        
        # 3. Force the column to be an integer to ensure absolute type safety
        raw_df[col] = raw_df[col].fillna(0).astype(int)

# Replace any remaining NaN values in other numeric/text columns with None for JSON
raw_df = raw_df.replace({np.nan: None})
# 3. Setup SageMaker runtime 
sm_runtime = boto3.client("sagemaker-runtime", region_name="ap-southeast-1")
ENDPOINT_NAME = "iti113-team14-airbnb-instant-booking"

def get_predictions_in_batches(df, batch_size=1000):
    all_probs = []
    records = df.to_dict(orient='records')
    total_records = len(records)
    
    print(f"Starting predictions for {len(records)} records...")
    
    for i in range(0, len(records), batch_size):
        batch = records[i : i + batch_size]
        payload = json.dumps({"data": batch})
        
        response = sm_runtime.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType="application/json",
            Accept="application/json",
            Body=payload
        )
        
        result = json.loads(response["Body"].read().decode("utf-8"))
        probs = [res["probability_class_1"] for res in result]
        all_probs.extend(probs)
        
         # Calculate overall progress
        processed = min(i + batch_size, total_records)
        percentage = (processed / total_records) * 100
        
        print(
            f"Processed {processed:,} / {total_records:,} records "
            f"({percentage:.1f}%)"
        )
        
    return all_probs

# 4. Generate probabilities and save your output
raw_df["probability_class_1"] = get_predictions_in_batches(raw_df)

clf_output = raw_df[['listing_id', 'probability_class_1']]
clf_output.to_csv('classification_predictions.csv', index=False)
print("Classification predictions saved successfully!")

# 5. UPLOAD the local file to S3 using your requested pattern
# --- 1.2 Version-control the raw file in S3 (skipped gracefully in local mode)
if not LOCAL_MODE:
    RAW_S3_URI = f's3://{BUCKET}/{PREFIX02}/raw/classification_predictions.csv'
    s3.upload_file(DATA_PATH, BUCKET, f'{PREFIX02}/raw/classification_predictions.csv')
    print(f'Raw data uploaded to: {RAW_S3_URI}')
else:
    print('LOCAL_MODE: skipping raw S3 upload (would go to '
          f's3://{BUCKET}/{PREFIX02}/raw/classification_predictions.csv)')

Starting predictions for 279712 records...


Processed 1,000 / 279,712 records (0.4%)


Processed 2,000 / 279,712 records (0.7%)


Processed 3,000 / 279,712 records (1.1%)


Processed 4,000 / 279,712 records (1.4%)


...

Processed 278,000 / 279,712 records (99.4%)


Processed 279,000 / 279,712 records (99.7%)


Processed 279,712 / 279,712 records (100.0%)


Classification predictions saved successfully!


Raw data uploaded to: s3://nyp-26s1-iti113/iti113/team14/data/airbnb-listings/raw/classification_predictions.csv


In [8]:
import pandas as pd
import boto3

# 1. Setup the S3 client and paths
s3_client = boto3.client("s3")
BUCKET = "nyp-26s1-iti113"
TEAM_ID = "team14"

# 2. Load the datasets
print("Loading master dataset and predictions...")
# Using raw_df assuming it is still in your notebook memory
reg_preds = pd.read_csv('regression_predictions.csv')

# Use the exact filename you referenced
clf_preds = pd.read_csv('classification_predictions.csv')

# Rename the column so it matches our recommendation system logic
clf_preds = clf_preds.rename(columns={'probability_class_1': 'instant_book_prob'})

# 3. Merge the datasets
print("Merging datasets...")
final_df = raw_df.merge(reg_preds, on='listing_id', how='inner')
final_df = final_df.merge(clf_preds, on='listing_id', how='inner')

# 4. Calculate Value Metric
# Positive residual = Overpriced. Negative residual = Underpriced (Good deal)
final_df['price_residual'] = final_df['price'] - final_df['pred_price']

print("\nMerge complete! Preview of the recommendation columns:")
# We ask for 'instant_book_prob' here because we just renamed it above!
print(final_df[['listing_id', 'price', 'pred_price', 'price_residual', 'instant_book_prob']].head())

# 5. Save the file LOCALLY first
local_filename = "master_recsys_dataset.csv"
final_df.to_csv(local_filename, index=False)
print(f"\n✅ Saved locally to {local_filename}")

# 6. UPLOAD the local file to S3 using your requested pattern
# We will upload it to a "processed" folder so we don't overwrite the original raw data
LOCAL_MODE = False # Ensure this is False so it actually uploads!

if not LOCAL_MODE:
    PROCESSED_S3_URI = f's3://{BUCKET}/{PREFIX99}/processed/master_recsys_dataset.csv'
    
    # 1. Use s3_client (not s3)
    # 2. Upload the local_filename (not DATA_PATH)
    s3_client.upload_file(
        local_filename, 
        BUCKET, 
        f'{PREFIX99}/processed/master_recsys_dataset.csv'
    )
    print(f'🚀 Fully scored dataset uploaded to: {PROCESSED_S3_URI}')
else:
    print('LOCAL_MODE: skipping S3 upload')

Loading master dataset and predictions...


Merging datasets...



Merge complete! Preview of the recommendation columns:
   listing_id  price  pred_price  price_residual  instant_book_prob
0      281420     53       67.11          -14.11           0.104575
1     3705183    120       67.05           52.95           0.104355
2     4082273     89       85.09            3.91           0.097841
3     4797344     58       74.25          -16.25           0.112057
4     4823489     60      105.48          -45.48           0.174739



✅ Saved locally to master_recsys_dataset.csv


🚀 Fully scored dataset uploaded to: s3://nyp-26s1-iti113/iti113/team14/data/airbnb-recommendation/processed/master_recsys_dataset.csv


## Recommendation Engine

In [9]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. Load your master dataset (from S3 or locally)
df = pd.read_csv("master_recsys_dataset.csv", low_memory=False)

# 2. Calculate and Normalize the "Value Score"
# (Predicted Price - Actual Price): Positive numbers mean it is a great deal
df['deal_value'] = df['pred_price'] - df['price']

scaler = MinMaxScaler()
# Scale between 0 and 1 so it matches your instant_book_prob scale
df['value_score'] = scaler.fit_transform(df[['deal_value']])

# 3. Define the Recommendation Engine
def recommend_airbnb(df, city, min_accommodates=1, max_price=None, w_value=0.5, w_convenience=0.5, top_n=5):
    
    # Step A: Candidate Generation (Apply Hard Filters)
    candidates = df[(df['city'].str.lower() == city.lower()) & 
                    (df['accommodates'] >= min_accommodates)].copy()
    
    if max_price is not None:
        candidates = candidates[candidates['price'] <= max_price]
        
    if candidates.empty:
        return "No listings match your criteria."

    # Step B: Apply the Machine Learning Ranking Formula
    candidates['final_score'] = (w_value * candidates['value_score']) + \
                                (w_convenience * candidates['instant_book_prob'])
                                
    # Step C: Sort by the highest score and return the results
    ranked_results = candidates.sort_values(by='final_score', ascending=False)
    
    columns_to_show = ['listing_id', 'city', 'accommodates', 'price', 
                       'pred_price', 'value_score', 'instant_book_prob', 'final_score']
                       
    return ranked_results[columns_to_show].head(top_n)

In [10]:
# 4. Test the Engine!
# Let's find a place in Paris for 2 people, under 150 EUR, favoring convenience (70% weight)
print("Top Recommendations:")
print(recommend_airbnb(
    df=df, 
    city="Paris", 
    min_accommodates=2, 
    max_price=150, 
    w_value=0.3,         # 30% emphasis on getting a good deal
    w_convenience=0.7,   # 70% emphasis on instant booking probability
    top_n=5
))

Top Recommendations:


        listing_id   city  accommodates  price  pred_price  value_score  \
136646    31369898  Paris             2    130      129.95     0.973260   
136020    22666855  Paris             2    119      129.04     0.973275   
142160    29872283  Paris             2    127      206.83     0.973384   
136647    35144515  Paris             2    128      145.57     0.973287   
135289    35177054  Paris             2     85      119.64     0.973314   

        instant_book_prob  final_score  
136646           0.999233     0.991441  
136020           0.999083     0.991341  
142160           0.998844     0.991206  
136647           0.998783     0.991134  
135289           0.998723     0.991100  


## Evaluation via Offline Proxy Metrics and Information Retrieval (IR) Metrics.

Overall Utility & Value Lift: Calculate the percentage improvement of your Top-10 ML results over a naive baseline (like sorting purely by lowest price).The objective is to demonstrate a substantial increase in booking convenience (a higher average Instant Book Probability) while simultaneously uncovering hidden financial value (a strong Price Residual discount) for the user.

NDCG@K (Normalized Discounted Cumulative Gain): NDCG evaluates ranking quality. You create an "ideal relevance" score for each candidate. NDCG measures how closely your engine's sorted output matches the mathematically perfect Top-K list.

## Utility Lift
standard price-sorting baseline Experiment for Utility Lift

In [11]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

def compare_recsys(df, city='Paris', min_accommodates=2, max_price=150, w_value=0.5, w_convenience=0.5):
    
    # 1. Candidate Generation (Apply Hard Filters)
    candidates = df[(df['city'].str.lower() == city.lower()) & 
                    (df['accommodates'] >= min_accommodates) &
                    (df['price'] <= max_price)].copy()
    
    if len(candidates) < 10:
        return "Not enough candidates to compare in this city."
        
    # 2. THE DUMMY BASELINE: How a basic website sorts (Cheapest First)
    baseline_top_10 = candidates.sort_values(by='price', ascending=True).head(10)
    
    # 3. YOUR ML ENGINE: Prepare the Scores
    # We want a "Good Deal" to have a high score. 
    # Since price_residual = Actual - Predicted, a NEGATIVE number is a good deal.
    # Let's create 'deal_value' where a higher positive number is better:
    candidates['deal_value'] = candidates['pred_price'] - candidates['price']
    
    # Normalize the deal value between 0 and 1 so it balances with instant_book_prob
    scaler = MinMaxScaler()
    candidates['scaled_value_score'] = scaler.fit_transform(candidates[['deal_value']])
    
    # Calculate final ML Score
    candidates['ml_score'] = (w_value * candidates['scaled_value_score']) + \
                             (w_convenience * candidates['instant_book_prob'])
                             
    ml_top_10 = candidates.sort_values(by='ml_score', ascending=False).head(10)
    
    # 4. CALCULATE LIFTS & IMPROVEMENTS
    base_conv = baseline_top_10['instant_book_prob'].mean()
    ml_conv = ml_top_10['instant_book_prob'].mean()
    conv_lift = ((ml_conv - base_conv) / base_conv) * 100 if base_conv > 0 else 0
    
    # Calculate Value Improvement (How much more discount does the ML engine find?)
    # A negative residual is a discount. We want the ML engine to have a more negative residual.
    base_residual = baseline_top_10['price_residual'].mean()
    ml_residual = ml_top_10['price_residual'].mean()
    value_improvement = base_residual - ml_residual
    
    # 5. GENERATE THE COMPARISON TABLE
    comparison_data = {
        "Metric (Top 10 Averages)": [
            "Average Actual Price", 
            "Average Instant Book Probability", 
            "Average Price Residual (Underpriced Amount)",
            "Overall Utility & Value Lift" # The new comparison row!
        ],
        "Dummy Baseline (Cheapest First)": [
            f"€ {baseline_top_10['price'].mean():.2f}",
            f"{base_conv * 100:.1f}%",
            f"€ {base_residual:.2f}",
            "---"
        ],
        "Your ML Engine (Value + Conv)": [
            f"€ {ml_top_10['price'].mean():.2f}",
            f"{ml_conv * 100:.1f}%",
            f"€ {ml_residual:.2f}",
            f"⭐ {conv_lift:.0f}% more convenient & finds €{value_improvement:.2f}/night in hidden value!"
        ]
    }
    
    compare_df = pd.DataFrame(comparison_data)
    
    print("--------------------------------------------------")
    print(f"🏆 EVALUATION RESULTS: {city.upper()} SEARCH")
    print(f"Budget: €{max_price} | Guests: {min_accommodates}")
    print("--------------------------------------------------\n")
    
    return compare_df

# Assuming 'final_df' is your merged master dataset
# Run the comparison
comparison_table = compare_recsys(final_df, city='Paris', min_accommodates=2, max_price=150)
display(comparison_table.style.set_properties(**{'text-align': 'left'}))

--------------------------------------------------
🏆 EVALUATION RESULTS: PARIS SEARCH
Budget: €150 | Guests: 2
--------------------------------------------------



,Metric (Top 10 Averages),Dummy Baseline (Cheapest First),Your ML Engine (Value + Conv)
0,Average Actual Price,€ 1.60,€ 119.10
1,Average Instant Book Probability,24.7%,95.9%
2,Average Price Residual (Underpriced Amount),€ -190.70,€ -152.58
3,Overall Utility & Value Lift,---,⭐ 289% more convenient & finds €-38.12/night in hidden value!


## NDCG@K (Normalized Discounted Cumulative Gain)
NDCG will measure how well your ML algorithm mimics the mathematically perfect "True Relevance" ranking, and compare it against the baseline.

In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import ndcg_score

def evaluate_ndcg(df, city='Paris', min_accommodates=2, max_price=150, k=10):
    
    # 1. Candidate Generation (Apply Hard Filters)
    candidates = df[(df['city'].str.lower() == city.lower()) & 
                    (df['accommodates'] >= min_accommodates) &
                    (df['price'] <= max_price)].copy()
    
    if len(candidates) < k:
        return "Not enough candidates to evaluate in this city."
        
    # 2. Define "True Relevance" (The Ground Truth)
    # This represents the absolute perfect ranking. 
    # We use the ACTUAL instant_bookable boolean (0 or 1) + Scaled Deal Value
    candidates['deal_value'] = candidates['pred_price'] - candidates['price']
    scaler = MinMaxScaler()
    candidates['scaled_value'] = scaler.fit_transform(candidates[['deal_value']])
    
    # True Relevance Score (Range: ~0 to 2)
    candidates['true_relevance'] = candidates['scaled_value'] + candidates['instant_bookable']
    
    # 3. Define ML Engine Score (Your Algorithm)
    # Uses your PREDICTED probability instead of the actual boolean
    w_value, w_convenience = 0.5, 0.5
    candidates['ml_score'] = (w_value * candidates['scaled_value']) + \
                             (w_convenience * candidates['instant_book_prob'])
                             
    # 4. Define the Dummy Baseline Score (Cheapest First)
    # We negate the price so the lowest price gets the mathematically highest score in the ranking
    candidates['baseline_score'] = -candidates['price']
    
    # 5. Calculate NDCG@K
    # Scikit-learn's ndcg_score expects arrays of shape (n_samples, n_labels)
    true_rel_array = np.asarray([candidates['true_relevance'].values])
    ml_score_array = np.asarray([candidates['ml_score'].values])
    baseline_score_array = np.asarray([candidates['baseline_score'].values])
    
    ml_ndcg = ndcg_score(true_rel_array, ml_score_array, k=k)
    baseline_ndcg = ndcg_score(true_rel_array, baseline_score_array, k=k)
    
    print("--------------------------------------------------")
    print(f"📈 RANKING EVALUATION: NDCG@{k}")
    print("--------------------------------------------------")
    print(f"Dummy Baseline (Cheapest First): {baseline_ndcg:.4f}")
    print(f"Your ML Recommendation Engine:   {ml_ndcg:.4f}")
    
    improvement = ((ml_ndcg - baseline_ndcg) / baseline_ndcg) * 100
    print(f"\nYour ML Engine ranks the best properties {improvement:.1f}% more accurately!")
    
    return ml_ndcg, baseline_ndcg

# Run the evaluation!
# Note: Ensure your 'instant_bookable' column is 1s and 0s (which we cleaned earlier)
evaluate_ndcg(final_df, city='Paris', min_accommodates=2, max_price=150, k=10)

--------------------------------------------------
📈 RANKING EVALUATION: NDCG@10
--------------------------------------------------
Dummy Baseline (Cheapest First): 0.3751
Your ML Recommendation Engine:   0.9478

Your ML Engine ranks the best properties 152.7% more accurately!


(0.9477811967597969, 0.3751206704728863)